# 🧠 CognitiveEye — End-to-End Deep Metric Learning Pipeline
### Principal AI Infrastructure Pipeline | ResNet-50 Backbone + Hard Triplet Mining
---
> **Runtime:** GPU (T4/A100 recommended) | **Dataset:** Tiny-ImageNet 200-class → 10-class subset | **Output:** `cognitive_eye_backbone.pth`

---
## 🔷 BLOCK 1 — Automated Dataset Ingestion & Architecture Diagram
---

In [ ]:
# ============================================================
# CELL 1.0 — Environment Bootstrap & Dependency Installation
# ============================================================
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

required = ['torch', 'torchvision', 'Pillow', 'tqdm', 'matplotlib', 'scikit-learn', 'scipy']
for pkg in required:
    pip_install(pkg)

print('✅ All dependencies verified.')

In [ ]:
# ============================================================
# CELL 1.1 — Master Imports
# ============================================================
import os
import sys
import time
import math
import random
import shutil
import zipfile
import urllib.request
import urllib.error
from pathlib import Path
from collections import defaultdict
from typing import List, Tuple, Optional, Dict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision.datasets import ImageFolder
from PIL import Image
from tqdm import tqdm

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Device ───────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU    : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'   PyTorch: {torch.__version__}')
print(f'   TorchV : {torchvision.__version__}')

In [ ]:
# ============================================================
# CELL 1.2 — Global Hyperparameters & Path Config
# ============================================================

# ── Paths ────────────────────────────────────────────────────
ROOT          = Path('/content') if os.path.exists('/content') else Path('.')
DATA_ROOT     = ROOT / 'data'
TRAIN_DIR     = DATA_ROOT / 'train'
VAL_DIR       = DATA_ROOT / 'val'
WEIGHTS_PATH  = ROOT / 'cognitive_eye_backbone.pth'
ZIP_PATH      = ROOT / 'model.zip'
TINYIMAGENET_URL = 'http://cs231n.stanford.edu/tiny-imagenet-200.zip'
TINYIMAGENET_ZIP = ROOT / 'tiny-imagenet-200.zip'
TINYIMAGENET_DIR = ROOT / 'tiny-imagenet-200'

# ── Model / Training ─────────────────────────────────────────
EMBEDDING_DIM  = 128        # Final L2-normalized embedding size
NUM_CLASSES    = 10         # Subset of Tiny-ImageNet classes we fine-tune on
IMAGE_SIZE     = 224        # Resize to ImageNet-standard
BATCH_SIZE     = 64         # Adjust if OOM (try 32 on 8 GB GPU)
NUM_EPOCHS     = 30
LR_INIT        = 3e-4
WEIGHT_DECAY   = 1e-2
TRIPLET_MARGIN = 0.3        # Margin α for triplet loss
SAMPLES_PER_CLASS = 6       # Images per class inside each batch (for mining)
NUM_WORKERS    = 2

print('📋 Configuration:')
config_items = [
    ('Embedding Dim', EMBEDDING_DIM),
    ('Num Classes',   NUM_CLASSES),
    ('Image Size',    IMAGE_SIZE),
    ('Batch Size',    BATCH_SIZE),
    ('Epochs',        NUM_EPOCHS),
    ('LR Init',       LR_INIT),
    ('Weight Decay',  WEIGHT_DECAY),
    ('Triplet Margin',TRIPLET_MARGIN),
]
for k, v in config_items:
    print(f'   {k:<22}: {v}')

In [ ]:
# ============================================================
# CELL 1.3 — Dataset Download, Extraction & Folder Structure
# ============================================================

def download_with_progress(url: str, dest: Path):
    """Download a URL to dest with a live progress bar."""
    if dest.exists():
        print(f'   ↩  Skipping download — {dest.name} already exists.')
        return
    print(f'   ⬇  Downloading {url}')
    try:
        def _reporthook(count, block_size, total_size):
            pct = min(count * block_size / total_size * 100, 100)
            bar = '█' * int(pct // 2) + '░' * (50 - int(pct // 2))
            print(f'\r   [{bar}] {pct:5.1f}%', end='', flush=True)
        urllib.request.urlretrieve(url, dest, reporthook=_reporthook)
        print(f'\n   ✅ Saved to {dest}')
    except urllib.error.URLError as e:
        print(f'\n   ❌ Download failed: {e}')
        raise


def extract_zip(zip_path: Path, extract_to: Path):
    if extract_to.exists():
        print(f'   ↩  Skipping extraction — {extract_to.name} already exists.')
        return
    print(f'   📦 Extracting {zip_path.name} → {extract_to} ...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        members = zf.namelist()
        for i, member in enumerate(members):
            zf.extract(member, extract_to.parent)
            if i % 5000 == 0:
                pct = i / len(members) * 100
                print(f'\r   Extracting ... {pct:5.1f}%', end='', flush=True)
    print(f'\n   ✅ Extracted.')


def build_val_structure(tiny_dir: Path):
    """
    Tiny-ImageNet val/ ships as flat images + val_annotations.txt.
    Reorganize into val/<class_id>/image.JPEG format so ImageFolder works.
    """
    val_src   = tiny_dir / 'val'
    val_img   = val_src / 'images'
    ann_file  = val_src / 'val_annotations.txt'
    if not ann_file.exists():
        print('   ⚠  val_annotations.txt not found — skipping restructure.')
        return
    # Read annotations: filename \t class_id \t ...
    mapping = {}
    with open(ann_file) as f:
        for line in f:
            parts = line.strip().split('\t')
            mapping[parts[0]] = parts[1]
    # Move images into per-class subdirs
    moved = 0
    for img_file, cls_id in mapping.items():
        cls_dir = val_src / cls_id / 'images'
        cls_dir.mkdir(parents=True, exist_ok=True)
        src = val_img / img_file
        dst = cls_dir / img_file
        if src.exists() and not dst.exists():
            shutil.move(str(src), str(dst))
            moved += 1
    print(f'   ✅ Restructured {moved} val images into per-class folders.')


def select_subset(tiny_dir: Path, out_train: Path, out_val: Path, n_classes: int = 10):
    """
    Copy the first n_classes from Tiny-ImageNet into DATA_ROOT/train and /val.
    """
    if out_train.exists() and out_val.exists():
        print('   ↩  Subset already built. Skipping.')
        return
    train_src = tiny_dir / 'train'
    val_src   = tiny_dir / 'val'

    # Collect all class directories from train
    all_classes = sorted([d.name for d in train_src.iterdir() if d.is_dir()])[:n_classes]
    print(f'   📂 Selected {n_classes} classes: {all_classes[:5]} ...')

    for split_name, src_root, dst_root in [('train', train_src, out_train),
                                            ('val',   val_src,   out_val)]:
        for cls in all_classes:
            if split_name == 'train':
                src_cls = src_root / cls / 'images'
            else:
                src_cls = src_root / cls / 'images'
            dst_cls = dst_root / cls
            if dst_cls.exists():
                continue
            if src_cls.exists():
                shutil.copytree(str(src_cls), str(dst_cls))
    print(f'   ✅ Subset copied → {out_train.parent}')


def verify_dataset(train_dir: Path, val_dir: Path):
    train_imgs = list(train_dir.rglob('*.JPEG')) + list(train_dir.rglob('*.jpg')) + list(train_dir.rglob('*.png'))
    val_imgs   = list(val_dir.rglob('*.JPEG'))   + list(val_dir.rglob('*.jpg'))   + list(val_dir.rglob('*.png'))
    train_cls  = [d.name for d in train_dir.iterdir() if d.is_dir()]
    val_cls    = [d.name for d in val_dir.iterdir()   if d.is_dir()]
    print('\n📊 Dataset Verification:')
    print(f'   Train classes : {len(train_cls)}')
    print(f'   Train images  : {len(train_imgs)}')
    print(f'   Val classes   : {len(val_cls)}')
    print(f'   Val images    : {len(val_imgs)}')
    assert len(train_cls) >= NUM_CLASSES, 'Not enough train classes!'
    assert len(train_imgs) > 0,           'No train images found!'
    assert len(val_imgs)   > 0,           'No val images found!'
    print('   ✅ Verification passed.')
    return train_cls


# ── Execute pipeline ─────────────────────────────────────────
print('\n🚀 STEP 1 — Downloading Tiny-ImageNet-200 ...')
download_with_progress(TINYIMAGENET_URL, TINYIMAGENET_ZIP)

print('\n🚀 STEP 2 — Extracting archive ...')
extract_zip(TINYIMAGENET_ZIP, TINYIMAGENET_DIR)

print('\n🚀 STEP 3 — Restructuring val/ split ...')
build_val_structure(TINYIMAGENET_DIR)

print('\n🚀 STEP 4 — Building 10-class subset ...')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
select_subset(TINYIMAGENET_DIR, TRAIN_DIR, VAL_DIR, n_classes=NUM_CLASSES)

print('\n🚀 STEP 5 — Verifying dataset ...')
CLASS_NAMES = verify_dataset(TRAIN_DIR, VAL_DIR)

In [ ]:
# ============================================================
# CELL 1.4 — ASCII Architecture Diagram
# ============================================================

DIAGRAM = r"""
╔══════════════════════════════════════════════════════════════════════════════╗
║            CognitiveEye — Deep Metric Learning Architecture                ║
║            ResNet-50 Backbone + Projection Head + Hard Triplet Mining       ║
╚══════════════════════════════════════════════════════════════════════════════╝

  RAW INPUT IMAGE
  [B × 3 × 224 × 224]   (B = batch, 3 = RGB channels)
        │
        ▼
  ┌─────────────────────────────────────────────────────┐
  │  DATA AUGMENTATION (train only)                     │
  │  ├─ RandomResizedCrop(224, scale=(0.4, 1.0))        │
  │  ├─ RandomHorizontalFlip(p=0.5)                     │
  │  ├─ ColorJitter(bright=0.4, contrast=0.4,           │
  │  │              sat=0.4, hue=0.1)                   │
  │  ├─ RandomGrayscale(p=0.2)                          │
  │  ├─ RandomErasing(p=0.3, scale=(0.02, 0.15))        │
  │  └─ Normalize(ImageNet mean/std)                    │
  └─────────────────────────────────────────────────────┘
        │
        ▼  [B × 3 × 224 × 224]
  ╔═════════════════════════════════════════════════════╗
  ║              RESNET-50 BACKBONE                     ║
  ║  ┌──────────────────────────────────────────────┐  ║
  ║  │  Conv1 + BN + ReLU + MaxPool                 │  ║
  ║  │  [B × 64 × 56 × 56]          ◄── FROZEN      │  ║
  ║  └──────────────────────────────────────────────┘  ║
  ║  ┌──────────────────────────────────────────────┐  ║
  ║  │  Layer1 — 3× Bottleneck Blocks               │  ║
  ║  │  [B × 256 × 56 × 56]         ◄── FROZEN      │  ║
  ║  └──────────────────────────────────────────────┘  ║
  ║  ┌──────────────────────────────────────────────┐  ║
  ║  │  Layer2 — 4× Bottleneck Blocks               │  ║
  ║  │  [B × 512 × 28 × 28]         ◄── FROZEN      │  ║
  ║  └──────────────────────────────────────────────┘  ║
  ║  ┌──────────────────────────────────────────────┐  ║
  ║  │  Layer3 — 6× Bottleneck Blocks               │  ║
  ║  │  [B × 1024 × 14 × 14]        ◄── UNFROZEN ✦  │  ║
  ║  └──────────────────────────────────────────────┘  ║
  ║  ┌──────────────────────────────────────────────┐  ║
  ║  │  Layer4 — 3× Bottleneck Blocks               │  ║
  ║  │  [B × 2048 × 7 × 7]          ◄── UNFROZEN ✦  │  ║
  ║  └──────────────────────────────────────────────┘  ║
  ╚═════════════════════════════════════════════════════╝
        │
        ▼  [B × 2048 × 7 × 7]
  ┌─────────────────────────────────────────────────────┐
  │  AdaptiveAvgPool2d(1×1)  →  Flatten                 │
  │  [B × 2048]                                         │
  └─────────────────────────────────────────────────────┘
        │
        ▼  [B × 2048]
  ╔═════════════════════════════════════════════════════╗
  ║         PROJECTION HEAD  (Fully Trainable)          ║
  ║  ┌──────────────────────────────────────────────┐  ║
  ║  │  Linear(2048 → 512) + BN + ReLU              │  ║
  ║  │  [B × 512]                                   │  ║
  ║  └──────────────────────────────────────────────┘  ║
  ║  ┌──────────────────────────────────────────────┐  ║
  ║  │  Dropout(p=0.3)                               │  ║
  ║  └──────────────────────────────────────────────┘  ║
  ║  ┌──────────────────────────────────────────────┐  ║
  ║  │  Linear(512 → 128)                           │  ║
  ║  │  [B × 128]                                   │  ║
  ║  └──────────────────────────────────────────────┘  ║
  ║  ┌──────────────────────────────────────────────┐  ║
  ║  │  L2 Normalize  →  Unit Hypersphere            │  ║
  ║  │  [B × 128]  ║v║₂ = 1.0                       │  ║
  ║  └──────────────────────────────────────────────┘  ║
  ╚═════════════════════════════════════════════════════╝
        │
        ▼  [B × 128]  ← EMBEDDING VECTORS
  ╔═════════════════════════════════════════════════════╗
  ║          HARD TRIPLET MINING ENGINE                 ║
  ║                                                     ║
  ║   For each anchor A in batch:                       ║
  ║   ┌──────────────────────────────────────────────┐  ║
  ║   │  Hard Positive P* = argmax d(A, P)           │  ║
  ║   │           where P shares class(A)            │  ║
  ║   │  (most dissimilar same-class sample)         │  ║
  ║   └──────────────────────────────────────────────┘  ║
  ║   ┌──────────────────────────────────────────────┐  ║
  ║   │  Hard Negative N* = argmin d(A, N)           │  ║
  ║   │           where class(N) ≠ class(A)          │  ║
  ║   │  (most similar different-class sample)       │  ║
  ║   └──────────────────────────────────────────────┘  ║
  ╚═════════════════════════════════════════════════════╝
        │
        ▼
  ┌─────────────────────────────────────────────────────┐
  │  TRIPLET MARGIN LOSS                                │
  │  L = max(d(A,P*) - d(A,N*) + α, 0)                 │
  │  α = 0.3  (margin)                                  │
  │  d = Euclidean distance on L2-normalized vectors    │
  └─────────────────────────────────────────────────────┘
        │
        ▼
  AdamW + CosineAnnealingLR → Backprop → Update Layer3, Layer4, Head

  OUTPUT: cognitive_eye_backbone.pth
  INFERENCE USE: Extract [B × 128] embedding → cosine/euclidean kNN search
"""

print(DIAGRAM)

---
## 🔷 BLOCK 2 — The Core Deep Feature Cortex (PyTorch Logic)
---

In [ ]:
# ============================================================
# CELL 2.1 — Custom Dataset & Balanced Batch Sampler
# ============================================================

class BalancedBatchSampler(Sampler):
    """
    Yields batches where each batch contains exactly `samples_per_class`
    images for each of `n_classes_per_batch` classes, which is essential
    for in-batch triplet mining.  Total batch size = n_classes × samples.
    """
    def __init__(
        self,
        dataset: ImageFolder,
        n_classes_per_batch: int,
        samples_per_class: int,
    ):
        super().__init__(dataset)
        self.dataset            = dataset
        self.n_classes_per_batch = n_classes_per_batch
        self.samples_per_class   = samples_per_class
        self.batch_size          = n_classes_per_batch * samples_per_class

        # Build class → list[index] mapping
        self.label_to_indices: Dict[int, List[int]] = defaultdict(list)
        for idx, (_, label) in enumerate(dataset.samples):
            self.label_to_indices[label].append(idx)
        self.labels = list(self.label_to_indices.keys())
        self.n_batches = len(dataset) // self.batch_size

    def __len__(self) -> int:
        return self.n_batches

    def __iter__(self):
        for _ in range(self.n_batches):
            # Sample n_classes_per_batch classes without replacement
            chosen_classes = random.sample(self.labels,
                                           min(self.n_classes_per_batch, len(self.labels)))
            batch_indices = []
            for cls in chosen_classes:
                pool = self.label_to_indices[cls]
                chosen = random.choices(pool, k=self.samples_per_class)  # with replacement if needed
                batch_indices.extend(chosen)
            random.shuffle(batch_indices)
            yield batch_indices

print('✅ BalancedBatchSampler defined.')

In [ ]:
# ============================================================
# CELL 2.2 — CognitiveEye Model: ResNet-50 + Projection Head
# ============================================================

class ProjectionHead(nn.Module):
    """
    Maps the 2048-d backbone output to a 128-d L2-normalized embedding.
    Architecture: Linear → BN → ReLU → Dropout → Linear → L2Norm
    """
    def __init__(self, in_dim: int = 2048, hidden_dim: int = 512, out_dim: int = 128,
                 dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, out_dim, bias=False),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.net(x)
        # L2 normalize onto unit hypersphere
        return F.normalize(emb, p=2, dim=1)


class CognitiveEyeNet(nn.Module):
    """
    Full model: ResNet-50 backbone (partially frozen) + ProjectionHead.

    Freeze strategy:
      - conv1, bn1, layer1, layer2  → frozen (gradient disabled)
      - layer3, layer4              → unfrozen (fine-grained adaptation)
      - avgpool + head              → fully trainable
    """
    def __init__(self, embedding_dim: int = 128, pretrained: bool = True,
                 dropout: float = 0.3):
        super().__init__()

        # ── Load pretrained ResNet-50 ──────────────────────
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        backbone = models.resnet50(weights=weights)

        # ── Decompose backbone into named segments ─────────
        self.stem   = nn.Sequential(backbone.conv1, backbone.bn1,
                                    backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.avgpool = backbone.avgpool    # AdaptiveAvgPool2d(1,1)

        # ── Freeze stem, layer1, layer2 ────────────────────
        frozen_modules = [self.stem, self.layer1, self.layer2]
        for module in frozen_modules:
            for param in module.parameters():
                param.requires_grad = False

        # ── Ensure layer3 & layer4 are trainable ──────────
        for module in [self.layer3, self.layer4]:
            for param in module.parameters():
                param.requires_grad = True

        # ── Projection head ────────────────────────────────
        self.head = ProjectionHead(
            in_dim=2048,
            hidden_dim=512,
            out_dim=embedding_dim,
            dropout=dropout,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)       # [B, 64, 56, 56]
        x = self.layer1(x)     # [B, 256, 56, 56]
        x = self.layer2(x)     # [B, 512, 28, 28]
        x = self.layer3(x)     # [B, 1024, 14, 14]
        x = self.layer4(x)     # [B, 2048, 7, 7]
        x = self.avgpool(x)    # [B, 2048, 1, 1]
        x = torch.flatten(x, 1)# [B, 2048]
        x = self.head(x)       # [B, 128], L2-normed
        return x

    def get_param_groups(self, lr_backbone: float, lr_head: float):
        """Return optimizer param groups with differentiated LRs."""
        return [
            {'params': list(self.layer3.parameters()) +
                        list(self.layer4.parameters()),
             'lr': lr_backbone},
            {'params': self.head.parameters(),
             'lr': lr_head},
        ]


# ── Instantiate & report ──────────────────────────────────────
model = CognitiveEyeNet(embedding_dim=EMBEDDING_DIM, pretrained=True).to(DEVICE)

total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params   = total_params - trainable_params

print('🧠 CognitiveEyeNet instantiated:')
print(f'   Total params    : {total_params:,}')
print(f'   Trainable       : {trainable_params:,}  ({trainable_params/total_params*100:.1f}%)')
print(f'   Frozen          : {frozen_params:,}  ({frozen_params/total_params*100:.1f}%)')

# Quick shape sanity check
with torch.no_grad():
    dummy = torch.randn(4, 3, 224, 224, device=DEVICE)
    out   = model(dummy)
    norm  = out.norm(dim=1)
    print(f'   Output shape    : {list(out.shape)}')
    print(f'   L2 norm (≈1.0)  : {norm.mean().item():.6f} ± {norm.std().item():.6f}')
print('   ✅ Forward pass OK.')

In [ ]:
# ============================================================
# CELL 2.3 — Hard Triplet Mining + Triplet Margin Loss
# ============================================================

def pairwise_euclidean_distances(embeddings: torch.Tensor) -> torch.Tensor:
    """
    Compute the full B×B pairwise Euclidean distance matrix for a batch
    of L2-normalized embeddings [B × D].

    Uses the identity:  ||a - b||² = ||a||² + ||b||² - 2·(a·bᵀ)
    For L2-normalized vectors, ||a||² = 1 for all a, so:
        ||a - b||² = 2 - 2·(a·bᵀ)
    We clamp to ≥ 0 before sqrt to handle floating-point negatives.
    """
    dot_product = torch.mm(embeddings, embeddings.t())          # [B, B]
    sq_norm     = dot_product.diag()                            # [B]
    distances   = sq_norm.unsqueeze(0) - 2.0 * dot_product + sq_norm.unsqueeze(1)
    distances   = distances.clamp(min=1e-12).sqrt()             # [B, B], non-negative
    return distances


class HardTripletLoss(nn.Module):
    """
    Batch-Hard Triplet Loss (Hermans et al., 2017 "In Defense of the Triplet Loss").

    For every anchor i in the batch:
      Hard Positive  P* = argmax_{j: label[j]=label[i], j≠i}  d(i, j)
      Hard Negative  N* = argmin_{k: label[k]≠label[i]}       d(i, k)

    Loss per anchor: max(d(i, P*) - d(i, N*) + margin, 0)
    Final loss: mean over all valid anchors.

    Args:
        margin (float): The triplet margin α.
        mining_type (str): 'hard' (default) or 'semihard'.
    """
    def __init__(self, margin: float = 0.3, mining_type: str = 'hard'):
        super().__init__()
        self.margin      = margin
        self.mining_type = mining_type

    def forward(
        self,
        embeddings: torch.Tensor,  # [B, D] L2-normalized
        labels:     torch.Tensor,  # [B]    integer class labels
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Returns:
            loss:      scalar mean triplet loss
            pos_dist:  mean hard-positive distance (diagnostic)
            neg_dist:  mean hard-negative distance (diagnostic)
        """
        B = embeddings.size(0)
        dist_mat = pairwise_euclidean_distances(embeddings)  # [B, B]

        # ── Build masks ────────────────────────────────────
        # pos_mask[i,j] = True if labels[i]==labels[j] AND i≠j
        labels_eq  = labels.unsqueeze(0) == labels.unsqueeze(1)  # [B, B]
        eye_mask   = ~torch.eye(B, dtype=torch.bool, device=embeddings.device)
        pos_mask   = labels_eq  & eye_mask    # same class, different sample
        neg_mask   = ~labels_eq               # different class

        # ── Hard Positive Mining ───────────────────────────
        # For each anchor, pick the farthest same-class sample
        # Mask invalid positions with -inf so they don't win argmax
        pos_dist_mat = dist_mat.clone()
        pos_dist_mat[~pos_mask] = -1.0          # impossible distance
        hard_pos_dist, _ = pos_dist_mat.max(dim=1)   # [B]

        # ── Hard Negative Mining ───────────────────────────
        # For each anchor, pick the closest different-class sample
        neg_dist_mat = dist_mat.clone()
        neg_dist_mat[~neg_mask] = float('inf')  # impossible distance
        hard_neg_dist, _ = neg_dist_mat.min(dim=1)   # [B]

        # ── Compute per-anchor loss ────────────────────────
        triplet_loss = F.relu(hard_pos_dist - hard_neg_dist + self.margin)  # [B]

        # Filter anchors that have at least one valid positive
        valid_mask = pos_mask.any(dim=1) & neg_mask.any(dim=1)
        if valid_mask.sum() == 0:
            # Edge case: batch has only one class — return zero loss
            zero = torch.tensor(0.0, device=embeddings.device, requires_grad=True)
            return zero, zero.detach(), zero.detach()

        loss     = triplet_loss[valid_mask].mean()
        pos_dist = hard_pos_dist[valid_mask].mean().detach()
        neg_dist = hard_neg_dist[valid_mask].mean().detach()

        return loss, pos_dist, neg_dist


# ── Smoke-test the loss ───────────────────────────────────────
criterion = HardTripletLoss(margin=TRIPLET_MARGIN)

with torch.no_grad():
    test_emb = F.normalize(torch.randn(32, EMBEDDING_DIM), p=2, dim=1).to(DEVICE)
    test_lbl = torch.randint(0, NUM_CLASSES, (32,)).to(DEVICE)
    l, pd, nd = criterion(test_emb, test_lbl)

print('✅ HardTripletLoss smoke-test:')
print(f'   Loss             : {l.item():.4f}')
print(f'   Mean Pos Dist    : {pd.item():.4f}')
print(f'   Mean Neg Dist    : {nd.item():.4f}')
print(f'   Separation gap   : {(nd - pd).item():.4f}  (should grow during training)')

---
## 🔷 BLOCK 3 — Intense Training & Convergence Loop
---

In [ ]:
# ============================================================
# CELL 3.1 — Data Transforms & DataLoaders
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Training Transforms ───────────────────────────────────────
# Heavy augmentation to simulate unstable real-world camera feeds:
#   RandomResizedCrop  → handles zoom / framing variation
#   ColorJitter        → lighting / white balance shifts
#   RandomGrayscale    → illumination robustness
#   GaussianBlur       → out-of-focus / motion blur
#   RandomErasing      → partial occlusion simulation

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.40, 1.00),
                                  ratio=(0.75, 1.333),
                                  interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.4,
            contrast=0.4,
            saturation=0.4,
            hue=0.1,
        )
    ], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=IMAGE_SIZE // 20 * 2 + 1, sigma=(0.1, 2.0))
    ], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15), ratio=(0.3, 3.3),
                              value='random', inplace=False),
])

# ── Validation Transforms (no augmentation) ───────────────────
val_transform = transforms.Compose([
    transforms.Resize(int(IMAGE_SIZE * 1.143),
                      interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ── Build Datasets ───────────────────────────────────────────
train_dataset = ImageFolder(root=str(TRAIN_DIR), transform=train_transform)
val_dataset   = ImageFolder(root=str(VAL_DIR),   transform=val_transform)

print(f'📂 Train dataset: {len(train_dataset)} images, {len(train_dataset.classes)} classes')
print(f'📂 Val   dataset: {len(val_dataset)} images,   {len(val_dataset.classes)} classes')

# ── Balanced Sampler for training (ensures mining quality) ────
n_classes_per_batch = min(NUM_CLASSES, 10)
balanced_sampler = BalancedBatchSampler(
    dataset=train_dataset,
    n_classes_per_batch=n_classes_per_batch,
    samples_per_class=SAMPLES_PER_CLASS,
)
effective_batch = n_classes_per_batch * SAMPLES_PER_CLASS
print(f'\n⚖️  Balanced Sampler:')
print(f'   Classes/batch   : {n_classes_per_batch}')
print(f'   Samples/class   : {SAMPLES_PER_CLASS}')
print(f'   Effective batch : {effective_batch}')
print(f'   Batches/epoch   : {len(balanced_sampler)}')

# ── DataLoaders ──────────────────────────────────────────────
train_loader = DataLoader(
    train_dataset,
    batch_sampler=balanced_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == 'cuda'),
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == 'cuda'),
    persistent_workers=(NUM_WORKERS > 0),
)

print(f'\n✅ DataLoaders ready.')

In [ ]:
# ============================================================
# CELL 3.2 — Optimizer, Scheduler & Validation Utilities
# ============================================================

# ── Differentiated learning rates ────────────────────────────
#   Backbone (Layer3+4): lower LR — preserve pretrained knowledge
#   Head: higher LR    — learn fast from scratch
LR_BACKBONE = LR_INIT * 0.1
LR_HEAD     = LR_INIT

param_groups = model.get_param_groups(lr_backbone=LR_BACKBONE, lr_head=LR_HEAD)

optimizer = AdamW(
    param_groups,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),
    eps=1e-8,
)

scheduler = CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-6,
)

print('⚙️  Optimizer & Scheduler:')
print(f'   Optimizer       : AdamW')
print(f'   LR backbone     : {LR_BACKBONE:.2e}')
print(f'   LR head         : {LR_HEAD:.2e}')
print(f'   Weight decay    : {WEIGHT_DECAY}')
print(f'   Scheduler       : CosineAnnealingLR  T_max={NUM_EPOCHS}  η_min=1e-6')


# ── Memory-Bank based kNN Validation ─────────────────────────
@torch.no_grad()
def build_memory_bank(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Pass the full validation set through the model and collect embeddings.
    Returns:
        bank_embeddings : [N, D]  L2-normalized embeddings
        bank_labels     : [N]     ground-truth class labels
    """
    model.eval()
    all_emb, all_lbl = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking=True)
        emb  = model(imgs)    # [B, D]
        all_emb.append(emb.cpu())
        all_lbl.append(labels)
    return torch.cat(all_emb, dim=0), torch.cat(all_lbl, dim=0)


@torch.no_grad()
def knn_validation(
    model:    nn.Module,
    loader:   DataLoader,
    device:   torch.device,
    k:        int = 5,
) -> Tuple[float, torch.Tensor, torch.Tensor]:
    """
    kNN Top-1 accuracy using cosine similarity on the validation set.
    Uses leave-one-out style: we compute kNN from the full bank.
    For speed, we do a single forward pass to build the bank,
    then evaluate each sample against the rest.

    Returns: (top1_accuracy, bank_embeddings, bank_labels)
    """
    bank_emb, bank_lbl = build_memory_bank(model, loader, device)
    N = bank_emb.size(0)

    # Similarity matrix [N, N] — cosine since embeddings are L2-normed
    sim_matrix = torch.mm(bank_emb, bank_emb.t())   # [N, N]

    # Exclude self-similarity by setting diagonal to -inf
    sim_matrix.fill_diagonal_(-float('inf'))

    # Top-k neighbors
    _, topk_indices = sim_matrix.topk(k, dim=1)      # [N, k]

    # Majority vote
    topk_labels = bank_lbl[topk_indices]              # [N, k]
    pred_labels, _ = topk_labels.mode(dim=1)          # [N]

    top1_acc = (pred_labels == bank_lbl).float().mean().item() * 100.0
    return top1_acc, bank_emb, bank_lbl


print('\n✅ Optimizer, scheduler, and validation utilities ready.')

In [ ]:
# ============================================================
# CELL 3.3 — Master Training Loop
# ============================================================

def format_time(seconds: float) -> str:
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    if h > 0:
        return f'{h}h{m:02d}m{s:02d}s'
    return f'{m:02d}m{s:02d}s'


def train_one_epoch(
    model:     nn.Module,
    loader:    DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device:    torch.device,
    epoch:     int,
    n_epochs:  int,
) -> Tuple[float, float, float]:
    """
    Run one full training epoch.
    Returns: (mean_loss, mean_pos_dist, mean_neg_dist)
    """
    model.train()
    running_loss = 0.0
    running_pos  = 0.0
    running_neg  = 0.0
    n_batches    = 0

    pbar = tqdm(
        loader,
        desc=f'  Epoch {epoch:02d}/{n_epochs}',
        leave=False,
        dynamic_ncols=True,
        bar_format='{l_bar}{bar:30}{r_bar}',
    )

    for batch_idx, (images, labels) in enumerate(pbar):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # ── Forward pass ──────────────────────────────────
        embeddings = model(images)                   # [B, 128]
        loss, pos_dist, neg_dist = criterion(embeddings, labels)

        # ── Backward pass ─────────────────────────────────
        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        # Gradient clipping — prevents exploding gradients in deep layers
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        # ── Accumulate stats ──────────────────────────────
        running_loss += loss.item()
        running_pos  += pos_dist.item()
        running_neg  += neg_dist.item()
        n_batches    += 1

        # Live tqdm update
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'P↑':   f'{pos_dist.item():.3f}',
            'N↓':   f'{neg_dist.item():.3f}',
            'gap':  f'{(neg_dist - pos_dist).item():.3f}',
        })

    if n_batches == 0:
        return 0.0, 0.0, 0.0

    return (running_loss / n_batches,
            running_pos  / n_batches,
            running_neg  / n_batches)


# ── Training state ───────────────────────────────────────────
best_val_top1 = 0.0
history = {
    'epoch': [], 'train_loss': [],
    'pos_dist': [], 'neg_dist': [], 'val_top1': []
}

HEADER = (
    f"\n{'─'*90}\n"
    f"{'Epoch':>7} {'Train Loss':>11} {'Avg Pos Dist':>13} "
    f"{'Avg Neg Dist':>13} {'Gap':>8} {'Val Top-1%':>11} {'LR':>10} {'Time':>8}\n"
    f"{'─'*90}"
)
print(HEADER)

total_train_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()

    # ── Train ─────────────────────────────────────────────────
    mean_loss, mean_pos, mean_neg = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE,
        epoch=epoch,
        n_epochs=NUM_EPOCHS,
    )

    # ── Validate ──────────────────────────────────────────────
    val_top1, _, _ = knn_validation(model, val_loader, DEVICE, k=5)

    # ── Step scheduler ────────────────────────────────────────
    scheduler.step()
    current_lr = optimizer.param_groups[1]['lr']  # head LR

    # ── Log ───────────────────────────────────────────────────
    gap        = mean_neg - mean_pos
    epoch_time = time.time() - epoch_start

    history['epoch'].append(epoch)
    history['train_loss'].append(mean_loss)
    history['pos_dist'].append(mean_pos)
    history['neg_dist'].append(mean_neg)
    history['val_top1'].append(val_top1)

    # Mark best epoch
    is_best = val_top1 > best_val_top1
    if is_best:
        best_val_top1 = val_top1
        torch.save({
            'epoch':            epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state':  optimizer.state_dict(),
            'scheduler_state':  scheduler.state_dict(),
            'val_top1':         val_top1,
            'embedding_dim':    EMBEDDING_DIM,
            'class_names':      CLASS_NAMES,
        }, WEIGHTS_PATH)

    star = ' ★ BEST' if is_best else ''
    print(
        f"{epoch:>7d} "
        f"{mean_loss:>11.4f} "
        f"{mean_pos:>13.4f} "
        f"{mean_neg:>13.4f} "
        f"{gap:>8.4f} "
        f"{val_top1:>10.2f}% "
        f"{current_lr:>10.2e} "
        f"{format_time(epoch_time):>8}"
        f"{star}"
    )

total_time = time.time() - total_train_start
print(f"{'─'*90}")
print(f"\n🏁 Training complete!")
print(f"   Total time      : {format_time(total_time)}")
print(f"   Best Val Top-1  : {best_val_top1:.2f}%")
print(f"   Saved to        : {WEIGHTS_PATH}")

In [ ]:
# ============================================================
# CELL 3.4 — Training Curves (Text + Optional Matplotlib)
# ============================================================

# ── Text summary ─────────────────────────────────────────────
print('\n📈 Training Summary:')
print(f"{'Epoch':>6} {'Loss':>8} {'POS d':>8} {'NEG d':>8} {'GAP':>8} {'Val%':>8}")
print('─' * 50)
for i, ep in enumerate(history['epoch']):
    gap = history['neg_dist'][i] - history['pos_dist'][i]
    print(
        f"{ep:>6d} "
        f"{history['train_loss'][i]:>8.4f} "
        f"{history['pos_dist'][i]:>8.4f} "
        f"{history['neg_dist'][i]:>8.4f} "
        f"{gap:>8.4f} "
        f"{history['val_top1'][i]:>8.2f}"
    )

# ── Matplotlib curves ────────────────────────────────────────
try:
    import matplotlib.pyplot as plt
    epochs = history['epoch']
    gaps   = [history['neg_dist'][i] - history['pos_dist'][i]
               for i in range(len(epochs))]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('CognitiveEye Training Curves', fontsize=14, fontweight='bold')

    # Loss
    axes[0].plot(epochs, history['train_loss'], 'b-o', markersize=3)
    axes[0].set_title('Triplet Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)

    # Distance separation
    axes[1].plot(epochs, history['pos_dist'], 'r-o', markersize=3, label='Hard Pos')
    axes[1].plot(epochs, history['neg_dist'], 'g-o', markersize=3, label='Hard Neg')
    axes[1].plot(epochs, gaps, 'k--o', markersize=3, label='Gap (↑ better)')
    axes[1].set_title('Distance Separation')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Euclidean Distance')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    # Validation accuracy
    axes[2].plot(epochs, history['val_top1'], 'm-o', markersize=3)
    axes[2].set_title('Val kNN Top-1 Accuracy')
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Accuracy (%)')
    axes[2].grid(True, alpha=0.3)
    axes[2].axhline(y=best_val_top1, color='orange', linestyle='--',
                    label=f'Best: {best_val_top1:.2f}%')
    axes[2].legend()

    plt.tight_layout()
    plt.savefig(str(ROOT / 'training_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('\n✅ Training curves saved to training_curves.png')
except Exception as e:
    print(f'⚠  Matplotlib plot skipped: {e}')

---
## 🔷 BLOCK 4 — Zero-Shot Verification & Output Archiver
---

In [ ]:
# ============================================================
# CELL 4.1 — Load Saved Weights & Verify Integrity
# ============================================================

print('🔍 Loading cognitive_eye_backbone.pth ...')
assert WEIGHTS_PATH.exists(), f'Weights file not found at {WEIGHTS_PATH}!'

checkpoint = torch.load(WEIGHTS_PATH, map_location=DEVICE)

# Instantiate a fresh model and load weights
inference_model = CognitiveEyeNet(
    embedding_dim=checkpoint['embedding_dim'],
    pretrained=False,   # We load our own weights, not ImageNet init
).to(DEVICE)

missing, unexpected = inference_model.load_state_dict(
    checkpoint['model_state_dict'], strict=True
)
inference_model.eval()

print(f'   ✅ Weights loaded from epoch {checkpoint["epoch"]} ')
print(f'   Validation Top-1 at save : {checkpoint["val_top1"]:.2f}%')
print(f'   Missing keys             : {missing}')
print(f'   Unexpected keys          : {unexpected}')
print(f'   Embedding dimension      : {checkpoint["embedding_dim"]}')
print(f'   Classes saved            : {checkpoint["class_names"][:5]} ...')

# ── Parameter checksum ───────────────────────────────────────
total_norm = sum(
    p.data.norm(2).item() ** 2
    for p in inference_model.parameters()
) ** 0.5
print(f'   Parameter L2 norm        : {total_norm:.4f}  (non-zero = weights loaded correctly)')

In [ ]:
# ============================================================
# CELL 4.2 — Build Full Validation Memory Bank
# ============================================================

print('🏦 Building memory bank from validation set ...')
t0 = time.time()

bank_embeddings, bank_labels = build_memory_bank(
    model=inference_model,
    loader=val_loader,
    device=DEVICE,
)

elapsed = time.time() - t0

print(f'   Bank embeddings : {list(bank_embeddings.shape)}  (N × D)')
print(f'   Bank labels     : {list(bank_labels.shape)}')
print(f'   Build time      : {elapsed:.2f}s')

# Per-class counts
print('\n   Class distribution in memory bank:')
unique_labels, counts = bank_labels.unique(return_counts=True)
class_id_to_name = {v: k for k, v in val_dataset.class_to_idx.items()}
for lbl, cnt in zip(unique_labels.tolist(), counts.tolist()):
    print(f'     [{lbl:>2}] {class_id_to_name.get(lbl, "?"):<20} : {cnt} samples')

# Norm sanity check (all should be ~1.0)
norms = bank_embeddings.norm(dim=1)
print(f'\n   Embedding L2 norms  — mean: {norms.mean():.6f}  std: {norms.std():.6f}')
print('   ✅ Memory bank ready.')

In [ ]:
# ============================================================
# CELL 4.3 — Zero-Shot Inference on a Random Validation Image
# ============================================================

def run_zero_shot_inference(
    model:           nn.Module,
    bank_embeddings: torch.Tensor,
    bank_labels:     torch.Tensor,
    class_id_to_name:Dict[int, str],
    val_dataset:     ImageFolder,
    device:          torch.device,
    top_k:           int = 5,
    query_idx:       Optional[int] = None,
) -> None:
    """
    Pick a random (or specified) image from the validation set,
    extract its 128-d embedding, and run kNN retrieval against
    the pre-built memory bank.  Prints a full textual report.
    """
    model.eval()

    # ── Select query image ────────────────────────────────
    if query_idx is None:
        query_idx = random.randint(0, len(val_dataset) - 1)
    img_tensor, true_label = val_dataset[query_idx]   # already transformed
    img_path = val_dataset.samples[query_idx][0]

    # ── Forward pass ──────────────────────────────────────
    t_start = time.time()
    with torch.no_grad():
        query_input = img_tensor.unsqueeze(0).to(device)  # [1, 3, 224, 224]
        query_emb   = model(query_input)                   # [1, 128]
    infer_ms = (time.time() - t_start) * 1000

    query_emb_cpu = query_emb.squeeze(0).cpu()            # [128]

    # ── Similarity search ─────────────────────────────────
    # Cosine similarity: sim = q · b^T  (both L2-normed, so range [-1, 1])
    sims = torch.mv(bank_embeddings, query_emb_cpu)        # [N]

    # Exclude the query itself if it's in the bank
    sims[query_idx] = -float('inf')

    top_sims, top_idxs = sims.topk(top_k)
    top_labels = bank_labels[top_idxs]

    # Majority vote prediction
    votes = defaultdict(float)
    for sim, lbl in zip(top_sims.tolist(), top_labels.tolist()):
        votes[lbl] += sim
    pred_label = max(votes, key=votes.get)
    pred_class = class_id_to_name.get(pred_label, '?')
    true_class = class_id_to_name.get(true_label, '?')
    correct    = (pred_label == true_label)

    # ── Report ────────────────────────────────────────────
    print('╔══════════════════════════════════════════════════════════════╗')
    print('║              ZERO-SHOT INFERENCE REPORT                     ║')
    print('╚══════════════════════════════════════════════════════════════╝')
    print(f'  Query index      : {query_idx}')
    print(f'  Image path       : .../{Path(img_path).name}')
    print(f'  Inference time   : {infer_ms:.2f} ms')
    print()
    print('  ── Embedding Vector (128-d, first 16 dims) ──')
    emb_preview = query_emb_cpu[:16].numpy()
    print('  [' + '  '.join(f'{v:+.4f}' for v in emb_preview) + '  ...]')
    print(f'  L2 norm          : {query_emb_cpu.norm().item():.8f}  (should be 1.0)')
    print()
    print(f'  ── Top-{top_k} Nearest Neighbors ──')
    print(f'  {"Rank":<6} {"Similarity":>12} {"Class ID":>10} {"Class Name":<25} {"Match?"}')
    print(f'  {"-"*65}')
    for rank, (sim_val, idx_val, lbl_val) in enumerate(
        zip(top_sims.tolist(), top_idxs.tolist(), top_labels.tolist()), start=1
    ):
        cls_name = class_id_to_name.get(lbl_val, '?')
        match    = '✓' if lbl_val == true_label else '✗'
        print(f'  {rank:<6} {sim_val:>12.6f} {lbl_val:>10} {cls_name:<25} {match}')

    print()
    print('  ── Prediction ──')
    print(f'  Ground Truth     : [{true_label}] {true_class}')
    print(f'  Predicted Class  : [{pred_label}] {pred_class}')
    verdict = '✅  CORRECT' if correct else '❌  INCORRECT'
    print(f'  Verdict          : {verdict}')
    print()
    print('  ── Weighted Votes ──')
    sorted_votes = sorted(votes.items(), key=lambda x: x[1], reverse=True)
    for lbl_v, score_v in sorted_votes:
        bar_len = int(score_v / max(v for _, v in sorted_votes) * 30)
        bar     = '█' * bar_len + '░' * (30 - bar_len)
        print(f'  [{lbl_v:>2}] {class_id_to_name.get(lbl_v, "?"):<20} [{bar}] {score_v:.4f}')
    print('╚══════════════════════════════════════════════════════════════╝')


# ── Run inference on 3 random images ─────────────────────────
for trial in range(3):
    print(f'\n🔬 Trial {trial + 1}/3:')
    run_zero_shot_inference(
        model=inference_model,
        bank_embeddings=bank_embeddings,
        bank_labels=bank_labels,
        class_id_to_name=class_id_to_name,
        val_dataset=val_dataset,
        device=DEVICE,
        top_k=5,
    )

In [ ]:
# ============================================================
# CELL 4.4 — Full Validation Report: Per-Class Accuracy
# ============================================================

print('📊 Running full validation kNN evaluation (k=1, 5, 10) ...')

@torch.no_grad()
def full_knn_report(
    bank_emb:  torch.Tensor,
    bank_lbl:  torch.Tensor,
    class_id_to_name: Dict[int, str],
    k_values:  List[int] = [1, 5, 10],
) -> None:
    N = bank_emb.size(0)
    sim_matrix = torch.mm(bank_emb, bank_emb.t())   # [N, N]
    sim_matrix.fill_diagonal_(-float('inf'))

    max_k   = max(k_values)
    _, topk_all = sim_matrix.topk(max_k, dim=1)     # [N, max_k]

    print(f'\n  {"Class":<25}', end='')
    for k in k_values:
        print(f'  kNN@{k:>2}%', end='')
    print(f'  {"Count":>7}')
    print('  ' + '─' * (25 + len(k_values) * 12 + 10))

    overall_results = {k: [] for k in k_values}

    unique_classes = bank_lbl.unique().tolist()
    for cls_id in sorted(unique_classes):
        cls_mask = (bank_lbl == cls_id)
        cls_indices = cls_mask.nonzero(as_tuple=True)[0]
        cls_name = class_id_to_name.get(cls_id, f'cls_{cls_id}')

        print(f'  {cls_name:<25}', end='')
        for k in k_values:
            top_k_neighbors = topk_all[cls_indices, :k]        # [n_cls, k]
            neighbor_labels = bank_lbl[top_k_neighbors]         # [n_cls, k]
            # Majority vote
            preds, _ = neighbor_labels.mode(dim=1)              # [n_cls]
            correct  = (preds == cls_id).float().mean().item() * 100
            overall_results[k].append(correct)
            print(f'  {correct:>7.1f}%', end='')
        print(f'  {cls_indices.size(0):>7}')

    print('  ' + '─' * (25 + len(k_values) * 12 + 10))
    print(f'  {"OVERALL":<25}', end='')
    for k in k_values:
        mean_acc = np.mean(overall_results[k])
        print(f'  {mean_acc:>7.1f}%', end='')
    print(f'  {N:>7}')


full_knn_report(
    bank_emb=bank_embeddings,
    bank_lbl=bank_labels,
    class_id_to_name=class_id_to_name,
    k_values=[1, 5, 10],
)
print('\n✅ Full validation report complete.')

In [ ]:
# ============================================================
# CELL 4.5 — Embedding Space Visualization (t-SNE / PCA)
# ============================================================

try:
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm

    print('🎨 Computing t-SNE projection of embedding space ...')
    emb_np = bank_embeddings.numpy()
    lbl_np = bank_labels.numpy()

    # First reduce to 50-d with PCA for speed, then t-SNE to 2-d
    n_components_pca = min(50, emb_np.shape[0] - 1, emb_np.shape[1])
    pca      = PCA(n_components=n_components_pca, random_state=SEED)
    emb_pca  = pca.fit_transform(emb_np)
    print(f'   PCA variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%  ({n_components_pca} dims)')

    tsne     = TSNE(n_components=2, random_state=SEED, perplexity=min(30, emb_np.shape[0]//4),
                    n_iter=1000, learning_rate='auto', init='pca')
    emb_2d   = tsne.fit_transform(emb_pca)

    unique_cls = sorted(set(lbl_np.tolist()))
    colors     = cm.get_cmap('tab10', len(unique_cls))

    fig, ax = plt.subplots(figsize=(10, 8))
    for cls_id in unique_cls:
        mask    = (lbl_np == cls_id)
        cls_name= class_id_to_name.get(cls_id, f'cls_{cls_id}')
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1],
                   c=[colors(cls_id)], label=cls_name,
                   alpha=0.7, s=30, edgecolors='none')

    ax.set_title('CognitiveEye — 128-d Embedding Space (t-SNE)', fontsize=13, fontweight='bold')
    ax.set_xlabel('t-SNE Dim 1'); ax.set_ylabel('t-SNE Dim 2')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig(str(ROOT / 'embedding_tsne.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('   ✅ t-SNE saved to embedding_tsne.png')

except ImportError as e:
    print(f'⚠  Visualization skipped (missing lib): {e}')
except Exception as e:
    print(f'⚠  Visualization error: {e}')

In [ ]:
# ============================================================
# CELL 4.6 — Output Archiver: Zip Weights & Download Trigger
# ============================================================
import subprocess
import os

print('📦 Archiving model artifacts ...')

# ── Collect files to archive ─────────────────────────────────
files_to_zip = [str(WEIGHTS_PATH)]

# Optionally include training curve PNGs if they were created
for extra_file in ['training_curves.png', 'embedding_tsne.png']:
    fp = ROOT / extra_file
    if fp.exists():
        files_to_zip.append(str(fp))

# ── Create zip via Python (compatible with both Colab & local) ─
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for filepath in files_to_zip:
        arcname = Path(filepath).name
        zf.write(filepath, arcname=arcname)
        print(f'   + {arcname}')

zip_size_mb = ZIP_PATH.stat().st_size / 1e6
print(f'\n   ✅ Archive created: {ZIP_PATH}  ({zip_size_mb:.1f} MB)')

# ── Verify archive integrity ──────────────────────────────────
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    bad = zf.testzip()
    if bad is None:
        print('   ✅ Archive integrity check: PASSED')
    else:
        print(f'   ❌ Archive integrity check: FAILED on {bad}')

# ── Browser download trigger (Google Colab only) ──────────────
try:
    from google.colab import files
    print('\n⬇️  Triggering browser download for Google Colab ...')
    files.download(str(ZIP_PATH))
    print('   ✅ Download triggered. Check your browser downloads folder.')
except ImportError:
    print('\nℹ️  Not running in Google Colab.')
    print(f'   Your archive is at: {ZIP_PATH.resolve()}')
    print('   To download in Colab: run `from google.colab import files; files.download("model.zip")`')
    # For local Jupyter: open file browser or copy path
    try:
        # Try to display a clickable link in Jupyter
        from IPython.display import display, FileLink
        display(FileLink(str(ZIP_PATH)))
    except Exception:
        pass
except Exception as e:
    print(f'   ⚠  Download trigger failed: {e}')
    print(f'   File location: {ZIP_PATH.resolve()}')

In [ ]:
# ============================================================
# CELL 4.7 — Final Pipeline Summary
# ============================================================

SUMMARY = f"""
╔══════════════════════════════════════════════════════════════════════════╗
║                 COGNITIVEEYE PIPELINE — COMPLETE SUMMARY               ║
╚══════════════════════════════════════════════════════════════════════════╝

  DATASET
  ├─ Source       : Tiny-ImageNet-200 (cs231n.stanford.edu)
  ├─ Subset       : {NUM_CLASSES} classes, ~500 train / ~50 val per class
  └─ Image size   : {IMAGE_SIZE}×{IMAGE_SIZE} (resized from 64×64 originals)

  MODEL
  ├─ Backbone     : ResNet-50 (ImageNet-pretrained weights)
  ├─ Frozen       : stem, layer1, layer2
  ├─ Unfrozen     : layer3, layer4  (fine-grained adaptation)
  ├─ Head         : Linear(2048→512) + BN + ReLU + Drop + Linear(512→128)
  └─ Output       : 128-d L2-normalized embedding on unit hypersphere

  TRAINING
  ├─ Loss         : Batch-Hard Triplet Margin Loss (margin α={TRIPLET_MARGIN})
  ├─ Mining       : Online Hard Positive + Hard Negative per anchor
  ├─ Optimizer    : AdamW  (LR backbone={LR_BACKBONE:.2e}, head={LR_HEAD:.2e})
  ├─ Scheduler    : CosineAnnealingLR  T_max={NUM_EPOCHS}  η_min=1e-6
  ├─ Grad clip    : max_norm=1.0
  └─ Epochs       : {NUM_EPOCHS}

  RESULTS
  ├─ Best Val kNN Top-1 : {best_val_top1:.2f}%
  └─ Saved checkpoint   : cognitive_eye_backbone.pth

  ARTIFACTS
  ├─ cognitive_eye_backbone.pth  — trained model weights + metadata
  ├─ model.zip                   — compressed archive for download
  ├─ training_curves.png         — loss / distance / accuracy plots
  └─ embedding_tsne.png          — 2-D t-SNE of 128-d embedding space

  INFERENCE
  ├─ Load checkpoint  → CognitiveEyeNet.load_state_dict()
  ├─ Extract 128-d embedding from any image
  └─ Match via cosine similarity against pre-built memory bank

╔══════════════════════════════════════════════════════════════════════════╗
║  ✅ Pipeline complete. cognitive_eye_backbone.pth is production-ready.  ║
╚══════════════════════════════════════════════════════════════════════════╝
"""

print(SUMMARY)